In [1]:
import h5py
import numpy as np
import torch
import torchvision

In [2]:
dataset_path = "/home/shinfang-2f/jay_pick_two_arms_and_hands_sim_20250523_085738/Pick_Right_Arm_and_Hand_Sim_2025-06-12_trimmed.hdf5"

device = "cuda"

In [3]:
with h5py.File(dataset_path, "r") as f:
    demo_key = list(f["data"].keys())[0]
    demo_group = f["data"][demo_key]
    min = f['data'][demo_key].attrs['action_min'],
    max = f['data'][demo_key].attrs['action_max']

In [6]:
min

(array([-6.28, -6.28, -3.14, -6.28, -6.28, -6.28]),)

In [7]:
max

array([6.28, 6.28, 3.14, 6.28, 6.28, 6.28])

In [3]:
with h5py.File(dataset_path, "r") as f:
    demo_key = list(f["data"].keys())[0]
    demo_group = f["data"][demo_key]
    obs_group = demo_group["obs"]
    obs_dict = {"qpos": obs_group["qpos"][0], "qvel": obs_group["qvel"][0]}
    obs_dict["images"] = obs_group["images"][0]
    

In [8]:
state = torch.from_numpy(np.concatenate([obs_dict['qpos'], obs_dict['qvel']]))
image = torch.from_numpy(obs_dict["images"])

# Convert to float32 with image from channel first in [0,255]
# to channel last in [0,1]
state = state.to(torch.float32)
image = image.to(torch.float32) / 255
image = image.permute(2, 0, 1)

# resize image to 350x350 with torchvision
image = torchvision.transforms.Resize(350)(image)
# # Send data tensors from CPU to GPU
state = state.to(device, non_blocking=True)
image = image.to(device, non_blocking=True)

state = state.unsqueeze(0)
image = image.unsqueeze(0)

# Create the policy input dictionary
observation = {
    "observation.state": state,
    "observation.image": image,
}

# Predict the next action with respect to the current observation
with torch.inference_mode():
    action = policy.select_action(observation)

In [9]:
image.shape

torch.Size([3, 350, 350])

In [9]:
obs_dict['qpos'].shape
obs_dict['qvel'].shape
obs_dict['images'].shape
# true_action.shape


(480, 480, 3)

In [ ]:
with h5py.File(dataset_path, "r") as f:
    demo_key = list(f["data"].keys())[0]
    demo_group = f["data"][demo_key]
    obs_group = demo_group["obs"]
    obs_dict = {"qpos": obs_group["qpos"][0], "qvel": obs_group["qvel"][0]}
    if policy_type == "visual_motor":
        if "images" not in obs_group:
            raise KeyError(
                "Images not found in dataset for visual-motor policy"
            )
        obs_dict["images"] = obs_group["images"][0]
    true_action = demo_group["actions"][0]
# Run inference
print("Input observation dict keys:", list(obs_dict.keys()))
pred_action = pipeline.predict(obs_dict)
# Compare with ground truth
print("Ground truth action shape:", true_action.shape)
print("Predicted action shape:", pred_action.shape)
print("Ground truth action (first 5):", true_action[:5])
print("Predicted action (first 5):", pred_action[:5])
print(
    "Action difference (mean abs):",
    np.abs(pred_action - true_action).mean(),
)
print(
    "Action difference (max abs):", np.abs(pred_action - true_action).max()
)


In [1]:
import torch
from lerobot.common.datasets.lerobot_dataset import LeRobotDataset
from pathlib import Path


In [10]:
import torch
from lerobot.common.datasets.lerobot_dataset import LeRobotDataset
from pathlib import Path

# The path to your generated dataset
# Using an absolute path is a good practice when loading local datasets
repo_id = "/home/shinfang-2f/jay_pick_two_arms_and_hands_sim_20250523_085738/IL/emily_right_sim_approach"
dataset_path = Path(repo_id)

print(f"Loading dataset from: {dataset_path.absolute()}")

# Load the entire dataset
# When loading a local dataset, the repo_id should be the path to the dataset directory.
dataset = LeRobotDataset(repo_id=str(dataset_path))

print("\n--- Custom Metadata ---")

# Access the metadata via the `meta.info` attribute
# This is a dictionary containing all the data from `meta/info.json`
metadata_info = dataset.meta.info

joint_names = metadata_info.get("joint_names")
action_min = metadata_info.get("action_min")
action_max = metadata_info.get("action_max")

if joint_names:
    print(f"joint_names: {joint_names}")
else:
    print("joint_names not found in metadata.")

if action_min:
    print(f"action_min: {action_min}")
else:
    print("action_min not found in metadata.")

if action_max:
    print(f"action_max: {action_max}")
else:
    print("action_max not found in metadata.") 

Loading dataset from: /home/shinfang-2f/jay_pick_two_arms_and_hands_sim_20250523_085738/IL/emily_right_sim_approach

--- Custom Metadata ---
joint_names: ['ra_shoulder_pan_joint', 'ra_shoulder_lift_joint', 'ra_elbow_joint', 'ra_wrist_1_joint', 'ra_wrist_2_joint', 'ra_wrist_3_joint']
action_min: [-0.1487570583440696, -0.3209634950965833, 0.30239940069283655, -0.1526652912425387, 0.20236842875268057, -0.3601632691492701]
action_max: [0.051784064549549313, -0.16732115274781634, 0.7127637316466897, 0.26746957924715264, 0.4375951031970371, -0.24389338846893827]


In [12]:
metadata_info.keys()

dict_keys(['codebase_version', 'robot_type', 'total_episodes', 'total_frames', 'total_tasks', 'total_videos', 'total_chunks', 'chunks_size', 'fps', 'splits', 'data_path', 'video_path', 'features', 'action_max', 'action_min', 'joint_names'])

In [4]:
dataset.meta

LeRobotDatasetMetadata({
    Repository ID: '/home/shinfang-2f/jay_pick_two_arms_and_hands_sim_20250523_085738/IL/emily_right_sim_approach',
    Total episodes: '3',
    Total frames: '783',
    Features: '['observation.image', 'observation.state', 'action', 'timestamp', 'frame_index', 'episode_index', 'index', 'task_index']',
})',

In [ ]:
# show metatdata

# Show dataset metadata
print("Dataset metadata:")
print(f"  Number of episodes: {dataset.num_episodes}")
print(f"  Number of frames: {dataset.num_frames}")
print(f"  Frames per second: {dataset.meta.fps}")
print(f"  Robot type: {dataset.meta.robot_type}")
print(f"  Camera keys: {dataset.meta.camera_keys}")


In [3]:
print(f"Number of episodes: {dataset.num_episodes}")
print(f"Number of frames: {dataset.num_frames}")

# You can access metadata like this:
print(f"Frames per second: {dataset.meta.fps}")
print(f"Robot type: {dataset.meta.robot_type}")
print(f"Camera keys: {dataset.meta.camera_keys}")

Number of episodes: 3
Number of frames: 783
Frames per second: 50
Robot type: UR10e
Camera keys: ['observation.image']


In [2]:
repo_id = "/home/shinfang-2f/jay_pick_two_arms_and_hands_sim_20250523_085738/IL/emily_right_sim_approach"
dataset_path = Path(repo_id)

print(f"Loading dataset from: {dataset_path.absolute()}")

# Load the entire dataset
# When loading a local dataset, the repo_id should be the path to the dataset directory.
dataset = LeRobotDataset(repo_id=str(dataset_path))

sample = dataset[0]

print("\nSample from the dataset:")
for key, value in sample.items():
    if isinstance(value, torch.Tensor):
        print(f"  {key}: tensor of shape {value.shape} and dtype {value.dtype}")
    else:
        print(f"  {key}: {value}")


Loading dataset from: /home/shinfang-2f/jay_pick_two_arms_and_hands_sim_20250523_085738/IL/emily_right_sim_approach

Sample from the dataset:
  observation.image: tensor of shape torch.Size([3, 480, 480]) and dtype torch.float32
  observation.state: tensor of shape torch.Size([12]) and dtype torch.float32
  action: tensor of shape torch.Size([6]) and dtype torch.float32
  timestamp: tensor of shape torch.Size([]) and dtype torch.float32
  frame_index: tensor of shape torch.Size([]) and dtype torch.int64
  episode_index: tensor of shape torch.Size([]) and dtype torch.int64
  index: tensor of shape torch.Size([]) and dtype torch.int64
  task_index: tensor of shape torch.Size([]) and dtype torch.int64
  task: move single arm (right) for approach a cone


In [5]:




# Get the first frame of the dataset

# You can also use it with a PyTorch DataLoader
dataloader = torch.utils.data.DataLoader(
    dataset,
    batch_size=4,
    shuffle=True,
)

print("\nTesting with DataLoader:")
for batch in dataloader:
    print(f"Batch observation.image shape: {batch['observation.image'].shape}")
    print(f"Batch observation.state shape: {batch['observation.state'].shape}")
    print(f"Batch action shape: {batch['action'].shape}")
    break 


Testing with DataLoader:
Batch observation.image shape: torch.Size([4, 3, 480, 480])
Batch observation.state shape: torch.Size([4, 12])
Batch action shape: torch.Size([4, 6])
